# COMET-STOmics Alignment & Cell Segmentation Transfer

This notebook demonstrates the full pipeline for:
1. Loading COMET cell segmentations (from Visiopharm GeoJSON)
2. Registering COMET DAPI to STOmics DAPI using VALIS
3. Warping COMET segmentation polygons to STOmics coordinate space
4. Loading STOmics gene expression (cellbin GEF)
5. Aggregating STOmics expression per COMET cell -> AnnData
6. Mapping COMET cells <-> STOmics cells
7. Validating alignment quality

**Pilot sample:** SO34 (MLA, chip A03979E2) - the only Endo/MLA sample with full data on disk.

**Validation sample:** SO4 (Ovarian, chip C03027C4) - clean Ovarian sample with cellbin data.

---

## 1. Setup & Configuration

In [ ]:
import sys
import json
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tifffile as tiff

# Add repo root to path
REPO_DIR = Path(r'T:/0_Organizational/Git/MDACC-STOmics-COMET-MSI')
sys.path.insert(0, str(REPO_DIR))

from comet.alignment_utils import (
    DefaultPaths, ALL_ENDO_MLA, ALL_SAMPLES, OVARIAN_WITH_STOMICS,
    SAMPLES_WITH_CELLBIN,
    check_sample_data, load_geojson, geojson_centroids,
    load_mld_as_geojson, compare_geojson_sources,
    extract_comet_dapi, setup_registration_inputs,
    run_valis_registration, warp_geojson_with_valis,
    warp_coordinates, get_comet_slide,
    load_stomics_cellbin_gef, load_stomics_h5ad,
    aggregate_expression_per_comet_cell,
    map_comet_to_stomics_cells,
    compute_alignment_metrics,
    plot_alignment_validation, plot_distance_distribution,
    validate_protein_gene_correlation, PROTEIN_GENE_MAP,
    run_alignment_pipeline,
)

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    datefmt='%H:%M:%S'
)

plt.rcParams['figure.figsize'] = [12, 8]
plt.rcParams['figure.dpi'] = 100

print('Imports OK')

In [ ]:
# ============= CONFIGURATION =============

# Pilot sample
SAMPLE_ID = 'SO34'
CHIP_ID = 'A03979E2'
DISEASE = 'MLA'

# Output directory
OUTPUT_DIR = Path('T:/Sammy Data/projects/out/comet_stomics_alignment')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Pipeline parameters
AGGREGATION_METHOD = 'nearest'  # 'nearest' (fast) or 'polygon' (precise)
MAX_DISTANCE = 50  # pixels - max distance for cell matching
DAPI_CHANNEL = 0   # DAPI channel index in COMET BS image

print(f'Sample: {SAMPLE_ID} ({DISEASE})')
print(f'Chip: {CHIP_ID}')
print(f'Output: {OUTPUT_DIR}')

## 2. Data Availability Check

In [ ]:
# Check all Endo/MLA samples
print('Endo/MLA Sample Data Availability:')
print('=' * 80)

all_status = []
for sid, info in ALL_ENDO_MLA.items():
    status = check_sample_data(sid, info['chip_id'], info.get('aligned', False))
    status['disease'] = info['disease']
    all_status.append(status)

status_df = pd.DataFrame(all_status)
display_cols = ['sample_id', 'chip_id', 'disease', 'geojson', 'comet_bs',
                'stomics_dapi', 'stomics_cellbin', 'stomics_h5ad']
print(status_df[display_cols].to_string(index=False))

# Highlight which samples can run the full pipeline
full_pipeline = status_df[
    status_df['geojson'] & status_df['comet_bs'] &
    status_df['stomics_dapi'] & (status_df['stomics_cellbin'] | status_df['stomics_h5ad'])
]
print(f'\nSamples with full data for pipeline: {list(full_pipeline["sample_id"])}')

In [ ]:
# Also check Ovarian samples with STOmics (for validation)
print('\nOvarian Samples with STOmics (for validation):')
print('=' * 80)

ov_status = []
for sid, info in OVARIAN_WITH_STOMICS.items():
    status = check_sample_data(sid, info['chip_id'], info.get('aligned', False))
    status['disease'] = 'Ovarian'
    ov_status.append(status)

ov_df = pd.DataFrame(ov_status)
print(ov_df[display_cols].to_string(index=False))

## 3. Load and Inspect COMET Segmentations (GeoJSON)

In [ ]:
# Load pre-exported GeoJSON
geojson_path = DefaultPaths.geojson_path(SAMPLE_ID, aligned=False)
print(f'GeoJSON: {geojson_path}')
print(f'Exists: {geojson_path.exists()}')

geojson_data = load_geojson(geojson_path)
n_cells = len(geojson_data['features'])
print(f'\nTotal cells: {n_cells:,}')

# Show sample feature
sample_feat = geojson_data['features'][0]
print(f'\nSample feature properties: {sample_feat["properties"]}')
print(f'Polygon rings: {len(sample_feat["geometry"]["coordinates"])}')
print(f'Exterior vertices: {len(sample_feat["geometry"]["coordinates"][0])}')

In [ ]:
# Extract centroids and compute statistics
centroids, labels = geojson_centroids(geojson_data)
areas = [f['properties']['area_px'] for f in geojson_data['features']]

print(f'Centroid coordinate ranges:')
print(f'  X: [{centroids[:, 0].min():.0f}, {centroids[:, 0].max():.0f}]')
print(f'  Y: [{centroids[:, 1].min():.0f}, {centroids[:, 1].max():.0f}]')

print(f'\nCell area statistics (pixels):')
print(f'  Mean: {np.mean(areas):.0f}')
print(f'  Median: {np.median(areas):.0f}')
print(f'  Min: {np.min(areas)}, Max: {np.max(areas)}')

# Scatter plot of cell centroids
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Centroid positions
axes[0].scatter(centroids[:, 0], centroids[:, 1], s=0.1, alpha=0.3)
axes[0].set_aspect('equal')
axes[0].set_title(f'{SAMPLE_ID}: COMET Cell Centroids (n={n_cells:,})')
axes[0].set_xlabel('X (pixels)')
axes[0].set_ylabel('Y (pixels)')

# Area distribution
axes[1].hist(areas, bins=100, alpha=0.7, color='steelblue', edgecolor='white')
axes[1].set_xlabel('Cell Area (pixels)')
axes[1].set_ylabel('Count')
axes[1].set_title('Cell Area Distribution')
axes[1].set_xlim(0, np.percentile(areas, 99))

plt.tight_layout()
plt.show()

## 3b. Compare Segmentation Sources: TIF Mask vs MLD

**Plan Step 1a:** Compare GeoJSON exported from the TIF mask (existing) vs extracted from MLD file
to determine polygon count, shape fidelity, and coordinate alignment between the two approaches.

In [ ]:
# Load MLD file and convert to GeoJSON
mld_path = DefaultPaths.mld_path(SAMPLE_ID, aligned=False)
tif_mask_path = DefaultPaths.tif_mask_path(SAMPLE_ID, aligned=False)

print(f'MLD file: {mld_path}')
print(f'  Exists: {mld_path.exists()}')
print(f'TIF mask: {tif_mask_path}')
print(f'  Exists: {tif_mask_path.exists()}')

# Get image dimensions from TIF mask for coordinate transform
if tif_mask_path.exists():
    v_img = tiff.imread(str(tif_mask_path))
    img_width = v_img.shape[2] if len(v_img.shape) > 2 else v_img.shape[1]
    img_height = v_img.shape[1] if len(v_img.shape) > 2 else v_img.shape[0]
    print(f'\nTIF mask shape: {v_img.shape}')
    print(f'Image dimensions: {img_width} x {img_height}')
    del v_img

# Load MLD as GeoJSON
if mld_path.exists():
    geojson_mld = load_mld_as_geojson(
        mld_path,
        image_width=img_width,
        image_height=img_height,
    )
    print(f'\nMLD GeoJSON: {len(geojson_mld["features"])} features')
else:
    print('MLD file not found - skipping comparison')
    geojson_mld = None

In [ ]:
# Compare TIF-exported GeoJSON vs MLD-extracted GeoJSON
if geojson_mld is not None:
    comparison = compare_geojson_sources(geojson_data, geojson_mld)

    print('=' * 60)
    print('SEGMENTATION SOURCE COMPARISON: TIF vs MLD')
    print('=' * 60)
    print(f'TIF cells: {comparison["n_cells_tif"]:,}')
    print(f'MLD cells: {comparison["n_cells_mld"]:,}')
    print(f'Difference: {comparison["n_diff"]:,} ({comparison["pct_diff"]:.1f}%)')

    if 'centroid_match_median_dist' in comparison:
        print(f'\nCentroid matching (TIF -> nearest MLD):')
        print(f'  Median distance: {comparison["centroid_match_median_dist"]:.1f}px')
        print(f'  Mean distance: {comparison["centroid_match_mean_dist"]:.1f}px')
        print(f'  Within 10px: {comparison["centroid_match_pct_within_10px"]:.1f}%')
        print(f'  Within 50px: {comparison["centroid_match_pct_within_50px"]:.1f}%')

    print(f'\nCoordinate ranges:')
    print(f'  TIF X: {comparison.get("tif_x_range", "N/A")}')
    print(f'  MLD X: {comparison.get("mld_x_range", "N/A")}')
    print(f'  TIF Y: {comparison.get("tif_y_range", "N/A")}')
    print(f'  MLD Y: {comparison.get("mld_y_range", "N/A")}')

    print(f'\nRecommendation: {comparison["recommendation"]}')

    # Visual comparison
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    # TIF centroids
    tif_c, _ = geojson_centroids(geojson_data)
    axes[0].scatter(tif_c[:, 0], tif_c[:, 1], s=0.1, alpha=0.3, c='blue')
    axes[0].set_aspect('equal')
    axes[0].set_title(f'TIF GeoJSON ({len(tif_c):,} cells)')

    # MLD centroids
    mld_c = []
    for f in geojson_mld['features']:
        geom = f['geometry']
        if geom and geom['type'] == 'Polygon':
            coords = np.array(geom['coordinates'][0])
            mld_c.append([np.mean(coords[:-1, 0]), np.mean(coords[:-1, 1])])
        elif geom and geom['type'] == 'Point':
            mld_c.append(geom['coordinates'])
    mld_c = np.array(mld_c)

    if len(mld_c) > 0:
        axes[1].scatter(mld_c[:, 0], mld_c[:, 1], s=0.1, alpha=0.3, c='red')
        axes[1].set_aspect('equal')
        axes[1].set_title(f'MLD GeoJSON ({len(mld_c):,} cells)')

        # Overlay
        axes[2].scatter(tif_c[:, 0], tif_c[:, 1], s=0.3, alpha=0.2, c='blue', label='TIF')
        axes[2].scatter(mld_c[:, 0], mld_c[:, 1], s=0.3, alpha=0.2, c='red', label='MLD')
        axes[2].set_aspect('equal')
        axes[2].legend()
        axes[2].set_title('Overlay')

    plt.suptitle(f'{SAMPLE_ID}: TIF vs MLD Segmentation Comparison', fontweight='bold')
    plt.tight_layout()
    plt.savefig(str(OUTPUT_DIR / f'{SAMPLE_ID}_tif_vs_mld_comparison.png'), dpi=150)
    plt.show()

    # Decision: which source to use
    print(f'\n=> Using TIF GeoJSON for pipeline (pre-exported, validated for 31 samples)')
else:
    print('MLD comparison skipped (MLD file not available)')

## 4. Inspect STOmics Data

In [ ]:
# Check STOmics files
stomics_dapi_path = DefaultPaths.stomics_dapi_path(CHIP_ID)
stomics_cellbin_path = DefaultPaths.stomics_cellbin_path(CHIP_ID)
stomics_h5ad_path = DefaultPaths.stomics_h5ad_path(CHIP_ID)

print(f'STOmics DAPI: {stomics_dapi_path}')
print(f'  Exists: {stomics_dapi_path.exists()}')
print(f'STOmics cellbin GEF: {stomics_cellbin_path}')
print(f'  Exists: {stomics_cellbin_path.exists()}')
print(f'STOmics h5ad (bin200): {stomics_h5ad_path}')
print(f'  Exists: {stomics_h5ad_path.exists()}')

# Load and inspect DAPI image
if stomics_dapi_path.exists():
    dapi = tiff.imread(str(stomics_dapi_path))
    print(f'\nDAPI shape: {dapi.shape}, dtype: {dapi.dtype}')
    
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(dapi, cmap='gray')
    ax.set_title(f'STOmics DAPI: {CHIP_ID}')
    plt.show()
    del dapi

In [ ]:
# Load STOmics cellbin gene expression
if stomics_cellbin_path.exists():
    stomics_adata = load_stomics_cellbin_gef(stomics_cellbin_path)
    print(f'\nSTOmics AnnData shape: {stomics_adata.shape}')
    print(f'obs columns: {list(stomics_adata.obs.columns)}')
    print(f'obsm keys: {list(stomics_adata.obsm.keys())}')
    
    if 'spatial' in stomics_adata.obsm:
        coords = stomics_adata.obsm['spatial']
        print(f'\nSpatial coordinate ranges:')
        print(f'  X: [{coords[:, 0].min():.0f}, {coords[:, 0].max():.0f}]')
        print(f'  Y: [{coords[:, 1].min():.0f}, {coords[:, 1].max():.0f}]')
    
    print(f'\nTop 20 genes: {list(stomics_adata.var_names[:20])}')
elif stomics_h5ad_path.exists():
    stomics_adata = load_stomics_h5ad(stomics_h5ad_path)
    print(f'\nSTOmics AnnData shape: {stomics_adata.shape}')
    print(f'(Note: this is bin200, not cellbin)')
else:
    print('No STOmics data available!')
    stomics_adata = None

## 5. VALIS Registration (COMET DAPI -> STOmics DAPI)

Register COMET DAPI to STOmics DAPI, preserving STOmics coordinates as the reference frame.
This step extracts the DAPI channel from the 48GB COMET OME-TIFF, then runs rigid + non-rigid registration.

In [ ]:
# Set up registration input directory
sample_dir = OUTPUT_DIR / f'{SAMPLE_ID}_{CHIP_ID}'
sample_dir.mkdir(parents=True, exist_ok=True)

comet_bs_path = DefaultPaths.comet_bs_path(SAMPLE_ID)
print(f'COMET BS: {comet_bs_path}')
print(f'  Exists: {comet_bs_path.exists()}')

src_dir, ref_img = setup_registration_inputs(
    SAMPLE_ID, CHIP_ID, sample_dir,
    comet_bs_path=comet_bs_path,
    stomics_dapi_path=stomics_dapi_path,
    dapi_channel=DAPI_CHANNEL
)

print(f'\nRegistration source dir: {src_dir}')
print(f'Reference image: {ref_img}')
print(f'Files in source dir:')
for f in sorted(src_dir.iterdir()):
    print(f'  {f.name} ({f.stat().st_size / 1e6:.1f} MB)')

In [ ]:
# Run VALIS registration
# This performs both rigid (affine) and non-rigid (deformable) registration
reg_dir = sample_dir / 'registration_output'

registrar = run_valis_registration(src_dir, reg_dir, ref_img)

# Print registration info
print('\nRegistered slides:')
for name, slide in registrar.slide_dict.items():
    print(f'  {name}: is_ref={slide.is_ref}')

In [ ]:
# Optionally save and visualize registered slides
registered_dir = sample_dir / 'registered_slides'
registered_dir.mkdir(parents=True, exist_ok=True)
registrar.warp_and_save_slides(str(registered_dir))

# Visualize registered images
reg_files = sorted(registered_dir.glob('*.tiff')) + sorted(registered_dir.glob('*.tif'))
print(f'Registered slide files:')
for f in reg_files:
    print(f'  {f.name}')

if len(reg_files) >= 2:
    fig, axes = plt.subplots(1, 2, figsize=(14, 7))
    for i, f in enumerate(reg_files[:2]):
        img = tiff.imread(str(f))
        if len(img.shape) > 2:
            img = img[0]  # First channel
        axes[i].imshow(img, cmap='gray')
        axes[i].set_title(f.name)
        del img
    plt.suptitle(f'{SAMPLE_ID}: Registered Slides', fontweight='bold')
    plt.tight_layout()
    plt.show()

## 6. Warp Cell Segmentations to STOmics Space

Apply the VALIS transformation to warp all COMET cell segmentation polygons from COMET coordinate space to STOmics coordinate space.

In [ ]:
# Warp GeoJSON cell segmentations
warped_path = sample_dir / f'{SAMPLE_ID}_warped_segmentations.geojson'

warped_geojson = warp_geojson_with_valis(
    registrar, geojson_path, warped_path
)

n_warped = len(warped_geojson['features'])
print(f'\nWarped {n_warped:,} cell features')
print(f'Saved to: {warped_path}')

In [ ]:
# Compare original vs warped coordinate ranges
orig_centroids, _ = geojson_centroids(geojson_data)
warped_centroids, _ = geojson_centroids(warped_geojson)

print('Coordinate Ranges:')
print(f'  Original (COMET space):')
print(f'    X: [{orig_centroids[:, 0].min():.0f}, {orig_centroids[:, 0].max():.0f}]')
print(f'    Y: [{orig_centroids[:, 1].min():.0f}, {orig_centroids[:, 1].max():.0f}]')
print(f'  Warped (STOmics space):')
print(f'    X: [{warped_centroids[:, 0].min():.0f}, {warped_centroids[:, 0].max():.0f}]')
print(f'    Y: [{warped_centroids[:, 1].min():.0f}, {warped_centroids[:, 1].max():.0f}]')

if stomics_adata is not None and 'spatial' in stomics_adata.obsm:
    st_coords = stomics_adata.obsm['spatial']
    print(f'  STOmics cells:')
    print(f'    X: [{st_coords[:, 0].min():.0f}, {st_coords[:, 0].max():.0f}]')
    print(f'    Y: [{st_coords[:, 1].min():.0f}, {st_coords[:, 1].max():.0f}]')

## 7. Aggregate STOmics Expression per COMET Cell

**Output A:** For each COMET cell (now in STOmics space), find the nearest STOmics data point and assign its gene expression. This creates an AnnData object with COMET cells x genes.

In [ ]:
if stomics_adata is not None:
    integrated_adata = aggregate_expression_per_comet_cell(
        warped_geojson, stomics_adata,
        method=AGGREGATION_METHOD,
        max_distance=MAX_DISTANCE
    )
    
    # Add metadata
    integrated_adata.uns['sample_id'] = SAMPLE_ID
    integrated_adata.uns['chip_id'] = CHIP_ID
    integrated_adata.uns['disease'] = DISEASE
    integrated_adata.uns['aggregation_method'] = AGGREGATION_METHOD
    integrated_adata.uns['max_distance'] = MAX_DISTANCE
    
    print(f'\nIntegrated AnnData: {integrated_adata.shape}')
    print(f'obs columns: {list(integrated_adata.obs.columns)}')
    
    # Save
    h5ad_path = sample_dir / f'{SAMPLE_ID}_{CHIP_ID}_comet_stomics_integrated.h5ad'
    integrated_adata.write_h5ad(str(h5ad_path))
    print(f'Saved: {h5ad_path}')
    
    # Statistics
    if 'nn_distance' in integrated_adata.obs.columns:
        dists = integrated_adata.obs['nn_distance'].values
        within = integrated_adata.obs['within_threshold'].sum()
        print(f'\nCells within {MAX_DISTANCE}px threshold: {within:,} / {len(dists):,} ({within/len(dists)*100:.1f}%)')
        print(f'Distance stats: min={dists.min():.1f}, median={np.median(dists):.1f}, max={dists.max():.1f}')
else:
    print('No STOmics data available - skipping aggregation')
    integrated_adata = None

## 8. COMET <-> STOmics Cell Mapping

**Output B:** Create a mapping between COMET cells and STOmics cells based on spatial proximity.

In [ ]:
if stomics_adata is not None:
    warped_centroids, comet_labels = geojson_centroids(warped_geojson)
    
    if 'spatial' in stomics_adata.obsm:
        st_centroids = stomics_adata.obsm['spatial']
    else:
        st_centroids = np.column_stack([
            stomics_adata.obs['x'].values, stomics_adata.obs['y'].values
        ])
    
    cell_mapping = map_comet_to_stomics_cells(
        warped_centroids, st_centroids, max_distance=MAX_DISTANCE
    )
    cell_mapping['comet_label'] = comet_labels
    
    # Save mapping
    mapping_path = sample_dir / f'{SAMPLE_ID}_{CHIP_ID}_cell_mapping.csv'
    cell_mapping.to_csv(mapping_path, index=False)
    print(f'\nSaved cell mapping: {mapping_path}')
    
    # Summary
    n_mapped = cell_mapping['mapped'].sum()
    print(f'\nMapping Summary:')
    print(f'  COMET cells: {len(cell_mapping):,}')
    print(f'  Mapped to STOmics: {n_mapped:,} ({n_mapped/len(cell_mapping)*100:.1f}%)')
    print(f'  Unmapped (>{MAX_DISTANCE}px): {len(cell_mapping) - n_mapped:,}')
    
    # Show mapping statistics
    mapped = cell_mapping[cell_mapping['mapped']]
    print(f'\nDistance distribution (mapped cells):')
    print(f'  min: {mapped["distance"].min():.1f}')
    print(f'  median: {mapped["distance"].median():.1f}')
    print(f'  mean: {mapped["distance"].mean():.1f}')
    print(f'  max: {mapped["distance"].max():.1f}')
else:
    print('No STOmics data - skipping cell mapping')
    cell_mapping = None

## 9. Alignment Validation

In [ ]:
# Compute alignment quality metrics
if stomics_adata is not None:
    metrics = compute_alignment_metrics(warped_centroids, st_centroids)
    
    print('=' * 60)
    print(f'ALIGNMENT QUALITY: {metrics["quality"]}')
    print('=' * 60)
    print(f'COMET cells: {metrics["n_comet_cells"]:,}')
    print(f'STOmics cells: {metrics["n_stomics_cells"]:,}')
    print(f'\nDistance to nearest STOmics cell:')
    print(f'  Median: {metrics["median_distance"]:.1f}px')
    print(f'  Mean: {metrics["mean_distance"]:.1f}px')
    print(f'  Std: {metrics["std_distance"]:.1f}px')
    print(f'  Range: [{metrics["min_distance"]:.1f}, {metrics["max_distance"]:.1f}]')
    print(f'\nWithin thresholds:')
    print(f'  <30px (excellent): {metrics["pct_within_30px"]:.1f}%')
    print(f'  <50px (good): {metrics["pct_within_50px"]:.1f}%')
    print(f'  <100px (acceptable): {metrics["pct_within_100px"]:.1f}%')
    
    # Save metrics
    metrics_path = sample_dir / f'{SAMPLE_ID}_{CHIP_ID}_alignment_metrics.json'
    with open(metrics_path, 'w') as f:
        json.dump(metrics, f, indent=2)
    print(f'\nSaved metrics: {metrics_path}')

In [ ]:
# Validation visualization
if stomics_adata is not None:
    fig = plot_alignment_validation(
        warped_geojson,
        stomics_adata=stomics_adata,
        stomics_dapi_path=stomics_dapi_path if stomics_dapi_path.exists() else None,
        output_path=sample_dir / f'{SAMPLE_ID}_{CHIP_ID}_alignment_validation.png',
        sample_n=5000,
        title=f'{SAMPLE_ID} ({DISEASE}): COMET-STOmics Alignment'
    )

In [ ]:
# Distance distribution
if stomics_adata is not None:
    from scipy.spatial import cKDTree
    tree = cKDTree(st_centroids)
    distances, _ = tree.query(warped_centroids, k=1)
    
    fig = plot_distance_distribution(
        distances,
        output_path=sample_dir / f'{SAMPLE_ID}_{CHIP_ID}_distance_distribution.png',
        title=f'{SAMPLE_ID}: COMET-STOmics Distance Distribution'
    )

In [ ]:
# Zoomed overlay of warped cells on STOmics
if stomics_adata is not None:
    from matplotlib.patches import Polygon as MplPolygon
    from matplotlib.collections import PatchCollection
    
    # Select a region to zoom into
    center_x = np.median(warped_centroids[:, 0])
    center_y = np.median(warped_centroids[:, 1])
    zoom_radius = 2000  # pixels
    
    fig, ax = plt.subplots(figsize=(12, 12))
    
    # Plot STOmics cells in zoom region
    in_region_st = (
        (st_centroids[:, 0] > center_x - zoom_radius) &
        (st_centroids[:, 0] < center_x + zoom_radius) &
        (st_centroids[:, 1] > center_y - zoom_radius) &
        (st_centroids[:, 1] < center_y + zoom_radius)
    )
    ax.scatter(st_centroids[in_region_st, 0], st_centroids[in_region_st, 1],
               s=3, alpha=0.5, c='blue', label='STOmics cells')
    
    # Plot warped COMET polygons in zoom region
    patches = []
    for feature in warped_geojson['features']:
        coords = np.array(feature['geometry']['coordinates'][0])
        cx, cy = np.mean(coords[:-1, 0]), np.mean(coords[:-1, 1])
        if (abs(cx - center_x) < zoom_radius and abs(cy - center_y) < zoom_radius):
            patches.append(MplPolygon(coords, closed=True))
    
    if patches:
        pc = PatchCollection(patches, alpha=0.2, edgecolor='red',
                              facecolor='lightcoral', linewidth=0.8)
        ax.add_collection(pc)
    
    ax.set_xlim(center_x - zoom_radius, center_x + zoom_radius)
    ax.set_ylim(center_y - zoom_radius, center_y + zoom_radius)
    ax.set_aspect('equal')
    ax.legend(fontsize=12)
    ax.set_title(f'{SAMPLE_ID}: Zoomed Overlay ({len(patches)} COMET cells)', fontsize=14)
    ax.set_xlabel('X (STOmics pixels)')
    ax.set_ylabel('Y (STOmics pixels)')
    
    plt.tight_layout()
    plt.savefig(str(sample_dir / f'{SAMPLE_ID}_{CHIP_ID}_zoomed_overlay.png'), dpi=150)
    plt.show()

## 9b. Protein-Gene Correlation Validation

Cross-validate the alignment by correlating COMET protein intensities (e.g., CD4, CD8, CD68)
with their corresponding STOmics gene expression. High Spearman correlations indicate
that the spatial alignment correctly maps cells with matching molecular profiles.

Requires COMET protein extraction data (from script01_comet_analysis.py).

In [ ]:
# Protein-gene correlation validation
# Load COMET protein intensities if available
comet_protein_path = Path(f'T:/Sammy Data/projects/out/out_comet/{SAMPLE_ID}_BS_protein_combined.parquet')
print(f'COMET protein data: {comet_protein_path}')
print(f'  Exists: {comet_protein_path.exists()}')

if comet_protein_path.exists() and integrated_adata is not None:
    comet_protein_df = pd.read_parquet(comet_protein_path)
    print(f'  Shape: {comet_protein_df.shape}')
    print(f'  Columns: {list(comet_protein_df.columns)}')

    # Ensure row alignment: both should have n_comet_cells rows
    # Truncate to the smaller set if needed
    n_min = min(len(comet_protein_df), integrated_adata.n_obs)
    comet_protein_df = comet_protein_df.iloc[:n_min]

    print(f'\nKnown protein-gene mappings:')
    for prot, gene in PROTEIN_GENE_MAP.items():
        has_prot = prot in comet_protein_df.columns
        has_gene = gene in integrated_adata.var_names
        status = 'OK' if (has_prot and has_gene) else ('no protein' if not has_prot else 'no gene')
        if has_prot and has_gene:
            print(f'  {prot:>12s} -> {gene:<10s}: {status}')

    corr_results = validate_protein_gene_correlation(
        integrated_adata, comet_protein_df
    )

    if len(corr_results) > 0:
        print(f'\nProtein-Gene Correlation Results:')
        print(corr_results.to_string(index=False))

        # Visualize top correlations
        n_plot = min(6, len(corr_results))
        if n_plot > 0:
            fig, axes = plt.subplots(2, 3, figsize=(15, 10))
            axes = axes.ravel()

            for i, (_, row) in enumerate(corr_results.head(n_plot).iterrows()):
                ax = axes[i]
                prot_vals = comet_protein_df[row['protein']].values[:n_min]
                gene_idx = list(integrated_adata.var_names).index(row['gene'])
                gene_vals = integrated_adata.X[:n_min, gene_idx]
                if hasattr(gene_vals, 'toarray'):
                    gene_vals = gene_vals.toarray().ravel()

                ax.scatter(prot_vals, gene_vals, s=1, alpha=0.1)
                ax.set_xlabel(f'{row["protein"]} (protein)')
                ax.set_ylabel(f'{row["gene"]} (gene)')
                ax.set_title(f'rho={row["spearman_rho"]:.3f}, p={row["p_value"]:.1e}')

            for i in range(n_plot, len(axes)):
                axes[i].set_visible(False)

            plt.suptitle(f'{SAMPLE_ID}: Protein-Gene Correlations', fontweight='bold')
            plt.tight_layout()
            plt.savefig(str(sample_dir / f'{SAMPLE_ID}_{CHIP_ID}_protein_gene_correlations.png'), dpi=150)
            plt.show()
    else:
        print('No matching protein-gene pairs found in the data.')
else:
    if not comet_protein_path.exists():
        print('COMET protein data not found - run script01_comet_analysis.py first')
    if integrated_adata is None:
        print('No integrated AnnData - skipping protein-gene validation')

## 10. Validation Run: Ovarian Sample (SO4)

Run the same pipeline on an Ovarian sample to validate pipeline robustness.

In [ ]:
# Run full pipeline on SO4 (Ovarian) for comparison
VALIDATION_SAMPLE = 'SO4'
VALIDATION_CHIP = 'C03027C4'

# Check data availability first
val_status = check_sample_data(VALIDATION_SAMPLE, VALIDATION_CHIP)
print(f'Validation sample {VALIDATION_SAMPLE} ({VALIDATION_CHIP}):')
for k, v in val_status.items():
    if isinstance(v, bool):
        print(f'  {k}: {"OK" if v else "MISSING"}')

can_run = (val_status['geojson'] and val_status['comet_bs'] and 
           val_status['stomics_dapi'] and 
           (val_status['stomics_cellbin'] or val_status['stomics_h5ad']))

if can_run:
    print('\nAll data available - running pipeline...')
else:
    print('\nMissing data - cannot run full pipeline')
    print('Skipping validation run')

In [ ]:
# Run the full pipeline using the convenience function
if can_run:
    val_results = run_alignment_pipeline(
        sample_id=VALIDATION_SAMPLE,
        chip_id=VALIDATION_CHIP,
        work_dir=OUTPUT_DIR,
        aggregation_method=AGGREGATION_METHOD,
        max_distance=MAX_DISTANCE,
        save_registered_slides=True
    )
    
    if val_results.get('alignment_metrics'):
        m = val_results['alignment_metrics']
        print(f'\n{VALIDATION_SAMPLE} Alignment: {m["quality"]} '
              f'(median={m["median_distance"]:.1f}px)')

In [ ]:
# Compare pilot (SO34) vs validation (SO4) alignment quality
if can_run and val_results.get('alignment_metrics') and 'metrics' in dir():
    comparison = pd.DataFrame([
        {'sample': f'{SAMPLE_ID} ({DISEASE})', **metrics},
        {'sample': f'{VALIDATION_SAMPLE} (Ovarian)', **val_results['alignment_metrics']}
    ])
    
    display_cols = ['sample', 'quality', 'n_comet_cells', 'n_stomics_cells',
                    'median_distance', 'pct_within_30px', 'pct_within_50px']
    print('\nAlignment Comparison:')
    print('=' * 80)
    print(comparison[display_cols].to_string(index=False))
    print('=' * 80)

## 11. Summary & Output Files

In [ ]:
print('=' * 70)
print(f'PIPELINE SUMMARY: {SAMPLE_ID} ({DISEASE}, {CHIP_ID})')
print('=' * 70)

print(f'\nCOMET cells: {n_cells:,}')
print(f'Warped cells: {n_warped:,}')

if stomics_adata is not None:
    print(f'STOmics cells: {stomics_adata.n_obs:,}')
    print(f'STOmics genes: {stomics_adata.n_vars:,}')

if integrated_adata is not None:
    print(f'\nIntegrated AnnData: {integrated_adata.shape}')
    print(f'  Method: {AGGREGATION_METHOD}')
    if 'within_threshold' in integrated_adata.obs.columns:
        n_valid = integrated_adata.obs['within_threshold'].sum()
        print(f'  Cells with valid expression: {n_valid:,}')

if cell_mapping is not None:
    n_mapped = cell_mapping['mapped'].sum()
    print(f'\nCell mapping: {n_mapped:,}/{len(cell_mapping):,} mapped')

if 'metrics' in dir():
    print(f'\nAlignment Quality: {metrics["quality"]}')
    print(f'  Median distance: {metrics["median_distance"]:.1f}px')

# List output files
print(f'\nOutput files in {sample_dir}:')
for f in sorted(sample_dir.rglob('*')):
    if f.is_file():
        size_mb = f.stat().st_size / 1e6
        rel_path = f.relative_to(sample_dir)
        print(f'  {rel_path} ({size_mb:.1f} MB)')

print('\n' + '=' * 70)
print('PIPELINE COMPLETE')
print('=' * 70)

## Next Steps

1. **If alignment looks good:** Proceed to batch processing of all Endo/MLA samples
2. **If alignment looks off:** Check VALIS registration parameters, try different feature detectors
3. **Missing STOmics data:** Locate/download cellbin data for remaining 14 Endo/MLA chips
4. **Export missing GeoJSON:** Run `script04_export_missing_geojson.py` for SO58, SO59
5. **Downstream analysis:** Use integrated AnnData for Endo vs MLA differential analysis